# 05 – Spark-optimalisering: Partisjonering, Caching og Broadcast Joins

Denne notebooken demonstrerer tre av de viktigste optimaliseringsteknikene i Spark:

| Teknikk | Når den hjelper |
|---|---|
| **Caching** | Samme DataFrame brukes flere ganger |
| **Broadcast join** | Én liten tabell joines med én stor tabell |
| **Repartition vs coalesce** | Kontrollere antall output-filer og shuffle-kostnad |

Vi bruker syntetiske data (500k ansattrader + 10 avdelinger) for å gjøre effektene målbare.

## 0. Oppsett og syntetiske data

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("05-spark-optimalisering")
    .config("spark.sql.shuffle.partitions", "8")   # lavt for lokal demo
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} klar — shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

In [ ]:
# ── Stor tabell: 500 000 ansatte ───────────────────────────────────────────
departments = ["engineering", "marketing", "sales", "hr", "finance",
               "legal", "product", "design", "data", "operations"]

employees = (
    spark.range(500_000)          # 500k rader med id 0–499999
    .withColumn("name",       F.concat(F.lit("emp_"), F.col("id")))
    .withColumn("dept_id",    (F.col("id") % 10).cast("integer"))
    .withColumn("salary",     (F.rand(seed=42) * 80_000 + 40_000).cast("integer"))
    .withColumn("hire_year",  (F.col("id") % 10 + 2015).cast("integer"))
)

# ── Liten tabell: 10 avdelinger ────────────────────────────────────────────
dept_data = [(i, departments[i], f"VP of {departments[i].title()}") for i in range(10)]
dept_lookup = spark.createDataFrame(dept_data, ["dept_id", "dept_name", "vp"])

print(f"Ansatte  : {employees.count():,} rader, {employees.rdd.getNumPartitions()} partisjoner")
print(f"Avdelinger: {dept_lookup.count()} rader, {dept_lookup.rdd.getNumPartitions()} partisjoner")

---
## 1. Caching — unngå å reberegne den samme DataFrame

Spark er **lazy**: uten caching beregnes hele pipeline på nytt for hvert action (`count`, `show`, osv.).
`cache()` lagrer resultatet i minnet slik at gjentatte actions er raske.

In [ ]:
# Uten cache — beregnes to ganger fra scratch
t0 = time.time()
_ = employees.filter(F.col("salary") > 100_000).count()
_ = employees.filter(F.col("salary") > 100_000).groupBy("dept_id").count().collect()
t_no_cache = time.time() - t0
print(f"Uten cache: {t_no_cache:.2f}s")

In [ ]:
# Med cache — beregnes én gang, deretter servert fra minnet
high_earners = employees.filter(F.col("salary") > 100_000).cache()
high_earners.count()   # trigger materialisering

t0 = time.time()
_ = high_earners.count()
_ = high_earners.groupBy("dept_id").count().collect()
t_cache = time.time() - t0
print(f"Med cache : {t_cache:.2f}s")
print(f"Speedup   : {t_no_cache / t_cache:.1f}x")

high_earners.unpersist()   # frigjør minne

**Tommelregel:** Cache en DataFrame hvis den brukes 2+ ganger og er kostbar å beregne.  
Husk å kalle `unpersist()` når du er ferdig — Spark frigjør ikke cache automatisk i en notebook.

---
## 2. Broadcast Join — unngå shuffle ved join med liten tabell

Ved en vanlig join shuffler Spark **begge** tabeller på join-nøkkelen — dyrt for store tabeller.  
Med broadcast join sendes den **lille tabellen til alle workers** slik at den store aldri flyttes.

### 2a. Query plan uten broadcast (sort-merge join)

In [ ]:
# Skru av auto-broadcast for å se den "naive" planen
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

join_no_broadcast = employees.join(dept_lookup, on="dept_id")

print("=== Query plan UTEN broadcast (sort-merge join) ===")
join_no_broadcast.explain()

In [ ]:
# Mål tid
t0 = time.time()
_ = join_no_broadcast.groupBy("dept_name").agg(F.avg("salary")).collect()
t_smj = time.time() - t0
print(f"Sort-merge join: {t_smj:.2f}s")

### 2b. Query plan med broadcast join

In [ ]:
# Gjenaktiver auto-broadcast og bruk F.broadcast() eksplisitt
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10 MB

join_broadcast = employees.join(F.broadcast(dept_lookup), on="dept_id")

print("=== Query plan MED broadcast join ===")
join_broadcast.explain()

In [ ]:
t0 = time.time()
result = join_broadcast.groupBy("dept_name").agg(
    F.count("*").alias("antall"),
    F.round(F.avg("salary"), 0).alias("snitt_lønn")
).orderBy("dept_name")
result.collect()
t_broadcast = time.time() - t0

print(f"Broadcast join  : {t_broadcast:.2f}s")
print(f"Sort-merge join : {t_smj:.2f}s")
print(f"Speedup         : {t_smj / t_broadcast:.1f}x")
result.show()

**Hva skjer i query planen?**
- Uten broadcast: `SortMergeJoin` — begge sider sorteres og shuffles
- Med broadcast: `BroadcastHashJoin` — kun hash-oppslag, ingen shuffle av stor tabell

**Tommelregel:** Bruk `F.broadcast()` når den lille tabellen er < 10–100 MB.  
Spark gjør dette automatisk via `spark.sql.autoBroadcastJoinThreshold` (standard: 10 MB).

---
## 3. Repartition vs Coalesce

Begge endrer antall partisjoner — men på fundamentalt forskjellige måter:

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Shuffle | **Ja** — full shuffle | **Nei** — kombinerer eksisterende partisjoner |
| Retning | Opp **eller** ned | Bare **ned** (færre partisjoner) |
| Datadistribusjon | Jevn | Kan bli ujevn |
| Brukstilfelle | Omfordele data jevnt, øke parallelisme | Redusere antall output-filer billig |

In [ ]:
print(f"Original partisjoner: {employees.rdd.getNumPartitions()}")

# repartition — full shuffle, jevn fordeling
t0 = time.time()
df_repart = employees.repartition(4)
df_repart.count()   # trigger
t_repart = time.time() - t0

# coalesce — ingen shuffle, kombinerer partisjoner lokalt
t0 = time.time()
df_coalesce = employees.coalesce(4)
df_coalesce.count()   # trigger
t_coalesce = time.time() - t0

print(f"\nrepartition(4) → {df_repart.rdd.getNumPartitions()} partisjoner  [{t_repart:.2f}s]")
print(f"coalesce(4)    → {df_coalesce.rdd.getNumPartitions()} partisjoner  [{t_coalesce:.2f}s]")

In [ ]:
# Vis datadistribusjon per partisjon
from pyspark.sql import functions as F

def partition_sizes(df, label):
    sizes = df.withColumn("partition", F.spark_partition_id()) \
               .groupBy("partition").count() \
               .orderBy("partition").collect()
    print(f"\n{label}:")
    for row in sizes:
        bar = "█" * (row["count"] // 20000)
        print(f"  Partisjon {row['partition']}: {row['count']:>7,} rader  {bar}")

partition_sizes(df_repart,   "repartition(4) — jevn fordeling (shuffle)")
partition_sizes(df_coalesce, "coalesce(4)    — ujevn fordeling (ingen shuffle)")

In [ ]:
# Query plan: coalesce har ingen Exchange-node (ingen shuffle)
print("=== coalesce(4) — ingen Exchange ===")
employees.coalesce(4).explain()

print("\n=== repartition(4) — Exchange (shuffle) ===")
employees.repartition(4).explain()

**Når bruke hva?**
- `coalesce(n)` — før `write` for å redusere antall output-filer (f.eks. fra 200 til 10)
- `repartition(n)` — når du trenger jevn fordeling, f.eks. før en tung aggregering
- `repartition(n, col)` — partisjonering på en kolonne for å samle relaterte rader på samme worker

---
## 4. Oppsummering: kombinert optimalisering

In [ ]:
# Realistisk pipeline med alle tre teknikkene:
# 1. Broadcast join med liten tabell
# 2. Cache mellomresultat som brukes to ganger
# 3. coalesce før skriving for å begrense antall Delta-filer

enriched = (
    employees
    .join(F.broadcast(dept_lookup), on="dept_id")
    .filter(F.col("hire_year") >= 2020)
    .select("id", "name", "dept_name", "salary", "hire_year")
    .cache()
)

# Bruk #1: statistikk
print("Statistikk per avdeling (ansatt >= 2020):")
enriched.groupBy("dept_name") \
    .agg(
        F.count("*").alias("antall"),
        F.round(F.avg("salary"), 0).alias("snitt_lønn")
    ) \
    .orderBy("dept_name") \
    .show()

# Bruk #2: topp 5 lønninger
print("Topp 5 lønn:")
enriched.orderBy(F.col("salary").desc()).limit(5).show()

enriched.unpersist()

In [ ]:
spark.stop()